In [ ]:
from pathlib import Path
import pandas as pd
import os
import re

In [ ]:
# Define the CSV file pairs to merge
csv_pairs = {
    'all_conditions_skeleton_analysis.csv': {
        'oct': './Output/Oct_2025_300mM_exp/all_conditions_skeleton_analysis.csv',
        'nov': './Output/Nov_2025_300mm_exp/all_conditions_skeleton_analysis.csv'
    },
    'All_Conditions_Ring_Measurements.csv': {
        'oct': './Output/Oct_2025_300mM_exp/All_Conditions_Ring_Measurements.csv',
        'nov': './Output/Nov_2025_300mm_exp/All_Conditions_Ring_Measurements.csv'
    },
    'All_Conditions_Node_Counts.csv': {
        'oct': './Output/Oct_2025_300mM_exp/All_Conditions_Node_Counts.csv',
        'nov': './Output/Nov_2025_300mm_exp/All_Conditions_Node_Counts.csv'
    }
}

print("CSV file pairs to merge:")
for name, paths in csv_pairs.items():
    print(f"  {name}")
    print(f"    Oct: {paths['oct']}")
    print(f"    Nov: {paths['nov']}")


def extract_batch_from_path(file_path):
    """Extract batch identifier (Oct or Nov) from file path."""
    if 'Oct' in file_path or 'oct' in file_path:
        return 'Oct'
    elif 'Nov' in file_path or 'nov' in file_path:
        return 'Nov'
    else:
        return 'Unknown'


def normalize_image_name(image_name):
    """
    Remove the 'Cropped_masked_' prefix if present to normalize image names.
    This ensures consistent matching across different CSV files.
    """
    prefix = 'Cropped_masked_'
    if image_name.startswith(prefix):
        return image_name[len(prefix):]
    return image_name


def parse_image_name(image_name):
    """
    Parse the image_name to extract components.
    Returns: dict with keys: source_image, pos, z, o
    """
    # First normalize the image name (remove Cropped_masked_ prefix if present)
    normalized_name = normalize_image_name(image_name)
    
    # Extract everything before '_pos' as the source image identifier
    pos_match = re.search(r'_pos(\d+)', normalized_name)
    if not pos_match:
        return None
    
    pos_num = pos_match.group(1)
    source_image = normalized_name[:pos_match.start()]
    
    # Extract z number - handle both 'z##' and 'z_##' patterns
    z_match = re.search(r'_z_?(\d+)', normalized_name)
    if not z_match:
        return None
    z_num = z_match.group(1)
    
    # Extract o number - it's the last number after z
    # Look for pattern after z: z_##_# or z##_#
    o_match = re.search(r'_z_?\d+_(\d+)', normalized_name)
    if o_match:
        o_num = o_match.group(1)
    else:
        o_num = '1'  # Default to o1 if not present
    
    return {
        'source_image': source_image,
        'pos': pos_num,
        'z': z_num,
        'o': o_num
    }


def create_updated_name(batch, condition, image_id, pos, z, o):
    """
    Create the updated image name following the pattern:
    [Batch]_[Condition]_[ImageID]_[Pos#]_[z#]_[o#]
    """
    return f"{batch}_{condition}_{image_id}_pos{pos}_z{z}_o{o}"


# Test the parsing function with sample names including the Cropped_masked_ prefix
test_names = [
    'Image_4_Airyscan_Processing_pos1_z23',
    'Image_4_Airyscan_Processing_pos1_z23_2',
    '300_mm_90_min_1_Airyscan_Processing_pos1_z_61_1',
    'Cropped_masked_300_mm_6_hr_Image_2_Airyscan_Processing_pos4_z_9_1'
]

print("\n" + "="*60)
print("Testing parser with normalization:")
print("="*60)
for name in test_names:
    normalized = normalize_image_name(name)
    result = parse_image_name(name)
    print(f"Original:   {name}")
    print(f"Normalized: {normalized}")
    print(f"Parsed:     {result}")
    print()

In [ ]:
# STEP 1: Merge Oct and Nov batches for each CSV file
print("\n" + "="*60)
print("STEP 1: Merging Oct and Nov batches...")
print("="*60)

merged_dataframes = {}

for csv_name, paths in csv_pairs.items():
    print(f"\nMerging: {csv_name}")
    
    # Read Oct batch
    oct_df = pd.read_csv(paths['oct'])
    print(f"  Oct batch: {len(oct_df)} rows")
    
    # Read Nov batch
    nov_df = pd.read_csv(paths['nov'])
    print(f"  Nov batch: {len(nov_df)} rows")
    
    # Add a batch column to each before merging
    oct_df['batch'] = 'Oct'
    nov_df['batch'] = 'Nov'
    
    # Merge (concatenate) the two dataframes
    merged_df = pd.concat([oct_df, nov_df], ignore_index=True)
    print(f"  Merged total: {len(merged_df)} rows")
    
    # Store the merged dataframe
    merged_dataframes[csv_name] = merged_df

print("\n" + "="*60)
print(f"Successfully merged {len(merged_dataframes)} CSV file pairs")
print("="*60)

In [ ]:
# STEP 2: Build global image ID mapping across ALL merged dataframes
print("\n" + "="*60)
print("STEP 2: Building global image ID mapping across all dataframes...")
print("="*60)

source_image_to_id = {}
image_counter = 1

# Collect all unique source images from all merged dataframes
all_source_images = set()

for csv_name, df in merged_dataframes.items():
    print(f"\nScanning: {csv_name}")
    
    if 'image_name' not in df.columns:
        print(f"  Warning: No 'image_name' column found, skipping...")
        continue
    
    unique_source_images = set()
    for img_name in df['image_name'].unique():
        parsed = parse_image_name(img_name)
        if parsed:
            unique_source_images.add(parsed['source_image'])
    
    all_source_images.update(unique_source_images)
    print(f"  Found {len(unique_source_images)} unique source images")

# Assign IDs to all unique source images (sorted for consistency)
print(f"\nAssigning ImageIDs to {len(all_source_images)} unique source images:")
for source_img in sorted(all_source_images):
    source_image_to_id[source_img] = f"Image{image_counter:02d}"
    print(f"  {source_img} -> {source_image_to_id[source_img]}")
    image_counter += 1

print(f"\nTotal unique source images: {len(source_image_to_id)}")

# Show a few examples to verify consistency
print("\n" + "="*60)
print("Verifying consistency across dataframes:")
print("="*60)
sample_source_imgs = list(source_image_to_id.keys())[:3]
for source_img in sample_source_imgs:
    print(f"\n{source_img} -> {source_image_to_id[source_img]}")
    for csv_name, df in merged_dataframes.items():
        if 'image_name' in df.columns:
            # Count how many rows have this source image
            count = 0
            for img_name in df['image_name']:
                parsed = parse_image_name(img_name)
                if parsed and parsed['source_image'] == source_img:
                    count += 1
            if count > 0:
                print(f"  Found in {csv_name}: {count} rows")

In [ ]:
# STEP 3: Apply the image ID mapping to all merged dataframes
print("\n" + "="*60)
print("STEP 3: Creating 'image_name_updated' column in all dataframes...")
print("="*60)

for csv_name, df in merged_dataframes.items():
    print(f"\nProcessing: {csv_name}")
    
    if 'image_name' not in df.columns:
        print(f"  Warning: No 'image_name' column found, skipping...")
        continue
    
    if 'condition' not in df.columns:
        print(f"  Warning: No 'condition' column found, skipping...")
        continue
    
    # Create the new column
    updated_names = []
    for idx, row in df.iterrows():
        img_name = row['image_name']
        condition = row['condition']
        batch = row['batch']  # We added this during merge
        
        parsed = parse_image_name(img_name)
        if parsed:
            source_img = parsed['source_image']
            image_id = source_image_to_id.get(source_img, 'ImageXX')
            if condition == '50mm':
                updated_name = create_updated_name(
                    batch=batch,
                    condition='50mm',
                    image_id=image_id,
                    pos=parsed['pos'],
                    z=parsed['z'],
                    o=parsed['o']
                )
            elif condition == '300mm_90min':
                updated_name = create_updated_name(
                    batch=batch,
                    condition='300mm90min',
                    image_id=image_id,
                    pos=parsed['pos'],
                    z=parsed['z'],
                    o=parsed['o']
                )
            elif condition == '300mm_6hr':
                updated_name = create_updated_name(
                    batch=batch,
                    condition='300mm6hr',
                    image_id=image_id,
                    pos=parsed['pos'],
                    z=parsed['z'],
                    o=parsed['o']
                )
            updated_names.append(updated_name)
        else:
            # If parsing failed, keep original name
            updated_names.append(img_name)
    
    # Add the new column
    df['image_name_updated'] = updated_names
    
    print(f"  ✓ Added 'image_name_updated' column")
    print(f"  Sample transformations:")
    
    # Show a few examples from Oct batch
    oct_samples = df[df['batch'] == 'Oct'].head(2)
    for _, row in oct_samples.iterrows():
        print(f"    Original: {row['image_name']}")
        print(f"    Updated:  {row['image_name_updated']}")
    
    # Show a few examples from Nov batch
    nov_samples = df[df['batch'] == 'Nov'].head(2)
    for _, row in nov_samples.iterrows():
        print(f"    Original: {row['image_name']}")
        print(f"    Updated:  {row['image_name_updated']}")
    
    # Update the stored dataframe
    merged_dataframes[csv_name] = df

print("\n" + "="*60)
print("All dataframes updated successfully!")
print("="*60)

In [ ]:
# STEP 4: Save the merged dataframes
print("\n" + "="*60)
print("STEP 4: Saving merged dataframes...")
print("="*60)

output_dir = './Output/Merged_Oct_Nov'
os.makedirs(output_dir, exist_ok=True)

for csv_name, df in merged_dataframes.items():
    output_path = os.path.join(output_dir, csv_name)
    df.to_csv(output_path, index=False)
    print(f"\n✓ Saved: {output_path}")
    print(f"  Total rows: {len(df)}")
    print(f"  Oct rows: {len(df[df['batch'] == 'Oct'])}")
    print(f"  Nov rows: {len(df[df['batch'] == 'Nov'])}")
    print(f"  Columns: {', '.join(df.columns)}")

print("\n" + "="*60)
print("DONE! All merged dataframes saved to:", output_dir)
print("="*60)

# Summary statistics
print("\n" + "="*60)
print("SUMMARY:")
print("="*60)
print(f"Total CSV files merged: {len(merged_dataframes)}")
print(f"Total unique source images: {len(source_image_to_id)}")
print(f"Output directory: {output_dir}")
print("\nGenerated files:")
for csv_name in merged_dataframes.keys():
    print(f"  - {csv_name}")

In [ ]:
# VERIFICATION: Check that ImageIDs are consistent across all dataframes
print("\n" + "="*60)
print("VERIFICATION: ImageID consistency check")
print("="*60)

# Pick a few source images and verify they have the same ImageID across all dataframes
verification_source_images = list(source_image_to_id.keys())[:5]

for source_img in verification_source_images:
    expected_image_id = source_image_to_id[source_img]
    print(f"\nChecking source image: {source_img}")
    print(f"  Expected ImageID: {expected_image_id}")
    
    found_in_files = []
    for csv_name, df in merged_dataframes.items():
        if 'image_name_updated' not in df.columns:
            continue
        
        # Find rows with this source image
        matching_rows = []
        for idx, row in df.iterrows():
            parsed = parse_image_name(row['image_name'])
            if parsed and parsed['source_image'] == source_img:
                matching_rows.append(row)
        
        if matching_rows:
            found_in_files.append(csv_name)
            # Verify all rows have the expected ImageID in their updated name
            sample_row = matching_rows[0]
            updated_name = sample_row['image_name_updated']
            
            # Extract ImageID from updated name
            image_id_match = re.search(r'_(Image\d+)_', updated_name)
            if image_id_match:
                found_image_id = image_id_match.group(1)
                status = "✓" if found_image_id == expected_image_id else "✗"
                print(f"    {status} {csv_name}: {found_image_id} ({len(matching_rows)} rows)")
                if status == "✗":
                    print(f"      ERROR: Expected {expected_image_id}, found {found_image_id}")
            else:
                print(f"    ✗ {csv_name}: Could not extract ImageID from '{updated_name}'")
    
    if not found_in_files:
        print(f"    Note: Not found in any dataframe")

print("\n" + "="*60)
print("Verification complete!")
print("="*60)